In [ ]:
from pathlib import Path
import sys
import sqlite3

import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
# Find repo root so local package imports work even when cwd is analysis/notebooks
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "quantum_jobs").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Could not find repo root containing quantum_jobs/")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from quantum_jobs.db.paths import DB_PATH as PROJECT_DB_PATH

db_path = Path(PROJECT_DB_PATH).resolve()
print(f"Using DB: {db_path}")
print(f"Exists: {db_path.exists()}")

if not db_path.exists():
    raise FileNotFoundError(
        f"Expected database does not exist: {db_path}. "
        "Refusing to create a blank SQLite DB."
    )

conn = sqlite3.connect(db_path)


In [ ]:
# Inspect available tables before running analysis queries
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn,
)
display(tables)

table_names = set(tables["name"])
if "job_snapshots" not in table_names:
    raise RuntimeError(
        f"job_snapshots table not found in database: {db_path}\n"
        f"Tables found: {sorted(table_names)}\n"
        "This usually means the notebook is connected to the wrong or blank SQLite DB."
    )


In [ ]:
def run_query(query: str) -> pd.DataFrame:
    return pd.read_sql(query, conn)


In [ ]:
query = """
SELECT
    company,
    COALESCE(pulled_date, substr(pulled_at, 1, 10)) AS date,
    COUNT(*) AS job_count
FROM job_snapshots
GROUP BY company, date
ORDER BY date, company
"""

df = run_query(query)
df.head()


In [ ]:
plt.figure(figsize=(10, 5))
for company, group in df.groupby("company"):
    plt.plot(group["date"], group["job_count"], marker="o", label=company)

plt.title("Jobs Over Time by Company")
plt.xlabel("Date")
plt.ylabel("Job Count")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()
